# DX 704 Week 6 Project

This project will develop a treatment plan for a fictious illness "Twizzleflu".
Twizzleflu is a mild illness caused by a virus.
The main symptoms are a mild fever, fidgeting, and kicking the blankets off the bed or couch.
Mild dehydration has also been reported in more severe cases.
These symptoms typically last 1-2 weeks without treatment.
Word on the internet says that Twizzleflu can be cured faster by drinking copious orange juice, but this has not been supported by evidence so far.
You will be provided with a theoretical model of Twizzleflu modeled as a Markov decision process.
Based on the model, you will compute optimal treatment plans to optimize different criteria, and compare patient discomfort with the different plans.

The full project description, a template notebook, and raw data are available on GitHub: [Project 6 Materials](https://github.com/bu-cds-dx704/dx704-project-06).

We will model Twizzleflu as a Markov decision process.
The model transition probabilities are provided in the file "twizzleflu-transitions.tsv" and the expected rewards are in "twizzleflu-rewards.tsv".
The goal for Twizzleflu is to minimize the expected discomfort of the patient which is expressed as negative rewards in the file.

## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Evaluate a Do Nothing Plan

One of the treatment actions is to do nothing.
Calculate the expected discomfort (not rewards) of a policy that always does nothing.

Hint: for this value calculation and later ones, use value iteration.
The analytical solution has difficulties in practice when there is no discount factor.

In [1]:
import numpy as np
import pandas as pd

rewards_df = pd.read_csv("twizzleflu-rewards.tsv", sep="\t")
transitions_df = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")

rewards_df.head()


,action,state,reward
0,do-nothing,exposed-1,0.0
1,do-nothing,exposed-2,0.0
2,do-nothing,exposed-3,0.0
3,do-nothing,symptoms-1,-0.5
4,do-nothing,symptoms-2,-1.0


In [2]:
# Identify states and actions

actions = rewards_df["action"].drop_duplicates().tolist()
states = rewards_df["state"].drop_duplicates().tolist()

action_index = {a: i for i, a in enumerate(actions)}
state_index = {s: i for i, s in enumerate(states)}


In [3]:
# Create the reward and transition matrices 

R = np.zeros((len(actions), len(states)))

P = np.zeros((len(actions), len(states), len(states)))

for row in rewards_df.itertuples(index=False):
    R[action_index[row.action], state_index[row.state]] = row.reward

for row in transitions_df.itertuples(index=False):
    P[
        action_index[row.action],
        state_index[row.state],
        state_index[row.next_state]
    ] = row.probability

assert np.allclose(P.sum(axis=2), 1)

gamma = 1.0

Save the expected discomfort by state to a file "do-nothing-discomfort.tsv" with columns state and expected_discomfort.

In [4]:
# Copy the value iteration functions

def compute_qT_once(R, P, gamma, v):
    return R + gamma * P @ v

def iterate_values_once(R, P, gamma, v):
    return np.max(compute_qT_once(R, P, gamma, v), axis=0)

def value_iteration(R, P, gamma, max_iterations=100000, tolerance=1e-12):
    v_old = np.zeros(R.shape[-1])

    for i in range(max_iterations):
        v_new = iterate_values_once(R, P, gamma, v_old)

        if np.max(np.abs(v_new - v_old)) < tolerance:
            return v_new

        v_old = v_new

    raise RuntimeError("Value iteration did not converge")


In [5]:
# Copy the iterative policy evaluation function 

def iterative_policy_evaluation(
    R, P, gamma, pi,
    max_iterations=100000,
    tolerance=1e-12
):
    n = R.shape[-1]

    R_pi = R[pi, np.arange(n)].reshape(1, n)
    P_pi = P[pi, np.arange(n), :].reshape(1, n, n)

    v_old = np.zeros(n)

    for i in range(max_iterations):
        v_new = iterate_values_once(R_pi, P_pi, gamma, v_old)

        if np.max(np.abs(v_new - v_old)) < tolerance:
            return v_new

        v_old = v_new

    raise RuntimeError("Policy evaluation did not converge")


In [6]:
# Evaluate doing nothing 

do_nothing_pi = np.full(
    len(states),
    action_index["do-nothing"],
    dtype=int
)

do_nothing_values = iterative_policy_evaluation(
    R, P, gamma, do_nothing_pi
)

print(pd.DataFrame({
    "state": states,
    "expected_discomfort": -do_nothing_values
}))


        state  expected_discomfort
0   exposed-1             3.413333
1   exposed-2             4.266667
2   exposed-3             5.333333
3  symptoms-1             6.666667
4  symptoms-2             5.000000
5  symptoms-3             1.666667
6   recovered            -0.000000


In [7]:
# Save Part 1

pd.DataFrame({
    "state": states,
    "expected_discomfort": -do_nothing_values
}).to_csv(
    "do-nothing-discomfort.tsv",
    sep="\t",
    index=False
)

Submit "do-nothing-discomfort.tsv" in Gradescope.

## Part 2: Compute an Optimal Treatment Plan

Compute an optimal treatment plan for Twizzleflu.
It should minimize the expected discomfort (maximize the rewards).

In [20]:
# Compute an Optimal Treatment Plan
minimum_discomfort_values = value_iteration(R, P, gamma)

minimum_discomfort_pi = np.argmax(
    compute_qT_once(R, P, gamma, minimum_discomfort_values),
    axis=0
)

print(pd.DataFrame({
    "state": states,
    "action": [actions[a] for a in minimum_discomfort_pi]
}))


        state      action
0   exposed-1     sleep-8
1   exposed-2     sleep-8
2   exposed-3     sleep-8
3  symptoms-1    drink-oj
4  symptoms-2    drink-oj
5  symptoms-3    drink-oj
6   recovered  do-nothing


Save the optimal actions for each state to a file "minimum-discomfort-actions.tsv" with columns state and action.

In [ ]:
pd.DataFrame({
    "state": states,
    "action": [actions[a] for a in minimum_discomfort_pi]
}).to_csv(
    "minimum-discomfort-actions.tsv",
    sep="\t",
    index=False
)

Submit "minimum-discomfort-actions.tsv" in Gradescope.

## Part 3: Expected Discomfort

Using your previous optimal policy, compute the expected discomfort for each state.

In [22]:
# Evaluate the optimal policy
minimum_discomfort_policy_values = iterative_policy_evaluation(
    R, P, gamma, minimum_discomfort_pi
)

print(pd.DataFrame({
    "state": states,
    "expected_discomfort": -minimum_discomfort_policy_values
}))


        state  expected_discomfort
0   exposed-1                 0.75
1   exposed-2                 1.50
2   exposed-3                 3.00
3  symptoms-1                 6.00
4  symptoms-2                 4.50
5  symptoms-3                 1.50
6   recovered                -0.00


Save your results in a file "minimum-discomfort-values.tsv" with columns state and expected_discomfort.

In [ ]:
# Save part 3

pd.DataFrame({
    "state": states,
    "expected_discomfort": -minimum_discomfort_policy_values
}).to_csv(
    "minimum-discomfort-values.tsv",
    sep="\t",
    index=False
)

Submit "minimum-discomfort-values.tsv" in Gradescope.

## Part 4: Minimizing Twizzleflu Duration

Modifiy the Markov decision process to minimize the days until the Twizzle flu is over.
To do so, change the reward function to always be -1 if the current state corresponds to being sick (must have symptoms, exposed does not count) and 0 otherwise.
To be clear, the action does not matter for this reward function.


In [24]:
# Create a new reward function
duration_rewards_df = rewards_df.copy()

duration_rewards_df["reward"] = (
    duration_rewards_df["state"]
    .str.startswith("symptoms-")
    .map({True: -1, False: 0})
)

R_duration = np.zeros_like(R)

for row in duration_rewards_df.itertuples(index=False):
    R_duration[
        action_index[row.action],
        state_index[row.state]
    ] = row.reward

print(duration_rewards_df)


        action       state  reward
0   do-nothing   exposed-1       0
1   do-nothing   exposed-2       0
2   do-nothing   exposed-3       0
3   do-nothing  symptoms-1      -1
4   do-nothing  symptoms-2      -1
5   do-nothing  symptoms-3      -1
6   do-nothing   recovered       0
7     drink-oj   exposed-1       0
8     drink-oj   exposed-2       0
9     drink-oj   exposed-3       0
10    drink-oj  symptoms-1      -1
11    drink-oj  symptoms-2      -1
12    drink-oj  symptoms-3      -1
13    drink-oj   recovered       0
14     sleep-8   exposed-1       0
15     sleep-8   exposed-2       0
16     sleep-8   exposed-3       0
17     sleep-8  symptoms-1      -1
18     sleep-8  symptoms-2      -1
19     sleep-8  symptoms-3      -1
20     sleep-8   recovered       0


Save your new reward function in a file "duration-rewards.tsv" in the same format as "twizzleflu-rewards.tsv".

In [ ]:
duration_rewards_df.to_csv(
    "duration-rewards.tsv",
    sep="\t",
    index=False
)

Submit "duration-rewards.tsv" in Gradescope.

## Part 5: Optimize for Shorter Twizzleflu

Compute an optimal policy to minimize the duration of Twizzleflu.

In [26]:
# Run value iteration using duration rewards

minimum_duration_values = value_iteration(
    R_duration, P, gamma
)

minimum_duration_pi = np.argmax(
    compute_qT_once(
        R_duration, P, gamma, minimum_duration_values
    ),
    axis=0
)

print(pd.DataFrame({
    "state": states,
    "action": [actions[a] for a in minimum_duration_pi]
}))


        state      action
0   exposed-1     sleep-8
1   exposed-2     sleep-8
2   exposed-3     sleep-8
3  symptoms-1  do-nothing
4  symptoms-2  do-nothing
5  symptoms-3  do-nothing
6   recovered  do-nothing


Save the optimal actions for each state to a file "minimum-duration-actions.tsv" with columns state and action.

In [ ]:
pd.DataFrame({
    "state": states,
    "action": [actions[a] for a in minimum_duration_pi]
}).to_csv(
    "minimum-duration-actions.tsv",
    sep="\t",
    index=False
)

Submit "minimum-duration-actions.tsv" in Gradescope.

## Part 6: Shorter Twizzleflu?

Compute the expected number of days sick for each state to a file.

In [28]:
# Evaluate the duration policy
minimum_duration_policy_values = iterative_policy_evaluation(
    R_duration, P, gamma, minimum_duration_pi
)

print(pd.DataFrame({
    "state": states,
    "expected_sick_days": -minimum_duration_policy_values
}))


        state  expected_sick_days
0   exposed-1            1.250000
1   exposed-2            2.500000
2   exposed-3            5.000000
3  symptoms-1           10.000000
4  symptoms-2            6.666667
5  symptoms-3            3.333333
6   recovered           -0.000000


Save the expected sick days for each state to a file "minimum-duration-days.tsv" with columns state and expected_sick_days.

In [29]:
pd.DataFrame({
    "state": states,
    "expected_sick_days": -minimum_duration_policy_values
}).to_csv(
    "minimum-duration-days.tsv",
    sep="\t",
    index=False
)

Submit "minimum-duration-days.tsv" in Gradescope.

## Part 7: Speed vs Pampering

Compute the expected discomfort using the policy to minimize days sick, and compare the results to the expected discomfort when optimizing to minimize discomfort.

Hint: Make sure that you are reporting on expected discomfort, not repeating the expected number of days.

In [30]:
# Evaluate both policies using the original rewards

speed_values = iterative_policy_evaluation(
    R, P, gamma, minimum_duration_pi
)

pampering_values = iterative_policy_evaluation(
    R, P, gamma, minimum_discomfort_pi
)

comparison = pd.DataFrame({
    "state": states,
    "speed_discomfort": -speed_values,
    "minimize_discomfort": -pampering_values
})

print(comparison)


        state  speed_discomfort  minimize_discomfort
0   exposed-1          0.833333                 0.75
1   exposed-2          1.666667                 1.50
2   exposed-3          3.333333                 3.00
3  symptoms-1          6.666667                 6.00
4  symptoms-2          5.000000                 4.50
5  symptoms-3          1.666667                 1.50
6   recovered         -0.000000                -0.00


Save the results to a file "policy-comparison.tsv" with columns state, speed_discomfort, and minimize_discomfort.

In [ ]:
comparison.to_csv(
    "policy-comparison.tsv",
    sep="\t",
    index=False
)

Submit "policy-comparison.tsv" in Gradescope.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

In [32]:
acknowledgements = pd.DataFrame([
    {
        "source": "DX704 Week 6 Examples",
        "reference": "video_implementing_iterative_policy_evaluation.ipynb",
        "usage": "Referenced iterative policy evaluation to calculate expected rewards under fixed treatment policies."
    },
    {
        "source": "DX704 Week 6 Examples",
        "reference": "video_implementing_value_iteration.ipynb",
        "usage": "Referenced value iteration to calculate optimal state values and treatment actions."
    },
    {
        "source": "DX704 Week 6 Examples",
        "reference": "video_implementing_policy_iteration.ipynb",
        "usage": "Referenced policy iteration concepts for understanding optimal treatment strategies."
    },
    {
        "source": "Python Libraries",
        "reference": "NumPy and pandas",
        "usage": "Used for matrix calculations, data processing, and exporting TSV files."
    }
])

acknowledgements.to_csv(
    "acknowledgements.tsv",
    sep="\t",
    index=False
)
